In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc


In [ ]:
# =========================
# INPUT
# =========================
H5AD = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/05b_trajectory_revisit/adata_CD8_subset_clusters_0_1_2_4.h5ad"

# =========================
# OUTPUT DIR (DE_DIR)
# =========================
# 你后续 driver gene 脚本用的是 DE_DIR 里这些 CSV
DE_DIR = "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/07_perturb_seq_T_cells"
os.makedirs(DE_DIR, exist_ok=True)

DE_4TO1   = os.path.join(DE_DIR, "DE_cluster1_vs_cluster4.full.csv")     # 4 -> 1 uses 1 vs 4
DE_4TO2   = os.path.join(DE_DIR, "DE_cluster2_vs_cluster4.full.csv")     # 4 -> 2 uses 2 vs 4
DE_4_RVSNR= os.path.join(DE_DIR, "DE_cluster4_R_vs_NR.full.csv")         # within cluster4: R vs NR
DE_1_RVSNR= os.path.join(DE_DIR, "DE_cluster1_R_vs_NR.full.csv")         # within cluster1: R vs NR

# =========================
# DE settings
# =========================
METHOD = "wilcoxon"
N_GENES = None   # None = keep all genes; or set e.g. 5000
USE_LAYER = "log1p_norm"  # 若 adata.layers 里没有该层，会自动回退到 adata.X

# =========================
# Column hints (optional)
# =========================
# 自动检测失败时，手动指定：
CLUSTER_COL = 'leiden_T_0.6'   # e.g. "leiden" or "cluster" or "leiden_cd8"
RESPONSE_COL = 'Respond'  # e.g. "Responder" or "response" or "RNR" or "is_responder"

print("DE_DIR:", DE_DIR)
print("Will write:", DE_4TO1, DE_4TO2, DE_4_RVSNR, DE_1_RVSNR, sep="\n  - ")


In [ ]:
ad = sc.read_h5ad(H5AD)
print(ad)


In [ ]:
print(ad.obs['Respond'].value_counts())
print(ad.obs['leiden_T_0.6'].value_counts())

In [ ]:
ad.X = np.log1p(ad.layers["norm_expr"])

In [ ]:
ad.obs["RNR"] = ad.obs[RESPONSE_COL].astype(str).map({
    "Responder": "R",
    "Non-Responder": "NR"
})

# 只保留 R/NR（安全）
ad = ad[ad.obs["RNR"].isin(["R", "NR"])].copy()

# 确保 cluster 是字符串
ad.obs["cluster"] = ad.obs[CLUSTER_COL].astype(str)

In [ ]:
# 1 vs 4
sc.tl.rank_genes_groups(
    ad, groupby="cluster", groups=["1"], reference="4",
    method="wilcoxon", use_raw=False, n_genes=ad.n_vars
)
df = sc.get.rank_genes_groups_df(ad, group="1")
df.to_csv(DE_4TO1, index=False)
print("saved:", DE_4TO1, "rows=", df.shape[0])

# 2 vs 4
sc.tl.rank_genes_groups(
    ad, groupby="cluster", groups=["2"], reference="4",
    method="wilcoxon", use_raw=False, n_genes=ad.n_vars
)
df = sc.get.rank_genes_groups_df(ad, group="2")
df.to_csv(DE_4TO2, index=False)
print("saved:", DE_4TO2, "rows=", df.shape[0])


In [ ]:
ad4 = ad[ad.obs["cluster"] == "4"].copy()

sc.tl.rank_genes_groups(
    ad4, groupby="RNR", groups=["R"], reference="NR",
    method="wilcoxon", use_raw=False, n_genes=ad4.n_vars
)
df = sc.get.rank_genes_groups_df(ad4, group="R")
df.to_csv(DE_4_RVSNR, index=False)
print("saved:", DE_4_RVSNR, "rows=", df.shape[0])


In [ ]:
ad1 = ad[ad.obs["cluster"] == "1"].copy()

sc.tl.rank_genes_groups(
    ad1, groupby="RNR", groups=["R"], reference="NR",
    method="wilcoxon", use_raw=False, n_genes=ad1.n_vars
)
df = sc.get.rank_genes_groups_df(ad1, group="R")
df.to_csv(DE_1_RVSNR, index=False)
print("saved:", DE_1_RVSNR, "rows=", df.shape[0])
